# Condition Prediction Colab

Notebook pentru **generare de predicții**, nu evaluare.

Colab face doar partea care cere GPU:
- încarcă modelul open-source, cu sau fără LoRA adapter;
- rulează pe `TEST_COMPLETE_PAIR_IDS`;
- salvează predicțiile în layout-ul repo-ului;
- exportă un zip cu predicțiile.

Evaluarea se rulează local cu `src/SOTA_EVALUATION/eval.py`, după ce aduci zip-ul în repo.

Schimbi doar `MODEL_PRESET` în celula de config:
- `qwen3_4b` pentru Qwen FT sintetic;
- `rollama3_8b` pentru RoLlama FT sintetic cu prompt zero-shot;
- `rollama3_8b_icl` pentru RoLlama base, fără adapter, cu același prompt + aceleași seed examples ca ICL API.

Pentru presetul ICL nu se încarcă adapter. Exemplele vin din `src/ICL/real_examples_manifest.tsv`, prin `claude_few_shot.build_system_prompt()`.


In [ ]:
# Dependencies for local-model prediction.
%pip -q install -U \
  "transformers>=4.57.0,<5" \
  "accelerate>=1.11.0" \
  "peft>=0.17.0" \
  "bitsandbytes>=0.45.0" \
  "jsonschema" \
  "sentencepiece" \
  "protobuf>=5.29.1,<6" \
  "anthropic"


In [ ]:
# Repo checkout / refresh.
import os
import subprocess
import sys
from pathlib import Path

# Build the URL from parts so notebook frontends cannot turn it into markdown.
REPO_URL = "".join(["https://", "github.com/", "SoftNestSol/", "Medical-Notes.git"])
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Medical-Notes")
UPDATE_REPO = True
# Set True in Colab when /content checkout is dirty and git pull fails.
# This deletes only the ephemeral Colab clone, not Drive backups.
FRESH_CLONE = False

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def run_checked(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:", " ".join(str(part) for part in cmd))
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    if result.stderr.strip():
        print(result.stderr.strip())
    return result

if IN_COLAB:
    os.chdir("/content")
    if FRESH_CLONE and REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    elif UPDATE_REPO:
        try:
            subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
        except subprocess.CalledProcessError as exc:
            print("git pull --ff-only failed; continuing with existing /content checkout.")
            print("If you need the latest notebook code, delete /content/Medical-Notes and rerun checkout.")
            print("error:", exc)
    os.chdir(REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "data_split.py").exists():
            os.chdir(candidate)
            break

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
SOTA = SRC / "SOTA_EVALUATION"
ICL = SRC / "ICL"
SCRIPTS = ROOT / "scripts"
for path in [SRC, SOTA, ICL, SCRIPTS]:
    sys.path.insert(0, str(path))

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
print("ROOT", ROOT)

In [ ]:
# Config.
from data_split import TEST_COMPLETE_PAIR_IDS

MODEL_PRESET = "rollama3_8b_icl"  # "qwen3_4b", "rollama3_8b", or "rollama3_8b_icl"

MODEL_CONFIGS = {
    "qwen3_4b": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507",
        "condition_name": "cond5_qwen3_4b_synth_claude_ft",
        "drive_run_prefix": "qwen3-4b-synth-claude",
        "chat_template_kwargs": {"enable_thinking": False},
        "prompt_preset": "sft",
        "use_adapter": True,
        "max_input_tokens": 4096,
    },
    "rollama3_8b": {
        "model_id": "OpenLLM-Ro/RoLlama3-8b-Instruct",
        "condition_name": "cond5_rollama3_8b_synth_claude_ft",
        "drive_run_prefix": "rollama3-8b-synth-claude",
        "chat_template_kwargs": {},
        "prompt_preset": "zero_shot",
        "use_adapter": True,
        "max_input_tokens": 4096,
    },
    "rollama3_8b_icl": {
        "model_id": "OpenLLM-Ro/RoLlama3-8b-Instruct",
        "condition_name": "cond3_rollama3_8b_real_icl",
        "drive_run_prefix": None,
        "chat_template_kwargs": {},
        "prompt_preset": "icl",
        "use_adapter": False,
        "max_input_tokens": 8192,
    },
}

if MODEL_PRESET not in MODEL_CONFIGS:
    raise ValueError(f"Unknown MODEL_PRESET={MODEL_PRESET!r}; choose one of {sorted(MODEL_CONFIGS)}")

MODEL_CONFIG = MODEL_CONFIGS[MODEL_PRESET]
MODEL_ID = MODEL_CONFIG["model_id"]
CONDITION_NAME = MODEL_CONFIG["condition_name"]
DRIVE_RUN_PREFIX = MODEL_CONFIG["drive_run_prefix"]
CHAT_TEMPLATE_KWARGS = MODEL_CONFIG["chat_template_kwargs"]
PROMPT_PRESET = MODEL_CONFIG["prompt_preset"]
USE_ADAPTER = MODEL_CONFIG["use_adapter"]

PRED_DIR = ROOT / "data" / "chiropractor_ro" / "predictions" / CONDITION_NAME
CONV_DIR = ROOT / "data" / "chiropractor_ro" / "conversations"

# Empty RUN_NAME = auto-pick latest Drive run containing an adapter folder and matching DRIVE_RUN_PREFIX.
# Set RUN_NAME explicitly if you want an exact run, e.g. "qwen3-4b-synth-claude-a100-20260623-1234".
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/Medical-Notes/ft-runs")
RUN_NAME = ""
LOCAL_ADAPTER_DIR = Path(f"/content/ft_adapter_{MODEL_PRESET}")

TEST_IDS = sorted(TEST_COMPLETE_PAIR_IDS, key=lambda x: int(x.replace("audio", "")))
MAX_INPUT_TOKENS = MODEL_CONFIG.get("max_input_tokens", 4096)
MAX_NEW_TOKENS = 1024
TRANSCRIPT_HEAD_TOKENS = 1536
# Remaining transcript budget is used for the tail. This preserves anamnesis + late exam/objective notes.
OVERWRITE_PREDICTIONS = True

# Export locations. Drive backup is best-effort; direct download is fallback.
PREDICTION_BACKUP_ROOT = Path("/content/drive/MyDrive/Medical-Notes/predictions")
DOWNLOAD_PREDICTIONS = True

print("MODEL_PRESET", MODEL_PRESET)
print("MODEL_ID", MODEL_ID)
print("CONDITION_NAME", CONDITION_NAME)
print("USE_ADAPTER", USE_ADAPTER)
print("DRIVE_RUN_PREFIX", DRIVE_RUN_PREFIX)
print("PROMPT_PRESET", PROMPT_PRESET)
print("MAX_INPUT_TOKENS", MAX_INPUT_TOKENS)
print("TEST_IDS", len(TEST_IDS), TEST_IDS)
print("PRED_DIR", PRED_DIR)


In [ ]:
# Locate/copy LoRA adapter from Drive when this preset uses one.
import shutil

adapter_source = None

if USE_ADAPTER:
    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped/failed:", repr(exc))

    def find_adapter_dir() -> Path:
        if RUN_NAME:
            candidate = DRIVE_RUN_ROOT / RUN_NAME / "adapter"
            if not candidate.exists():
                raise FileNotFoundError(candidate)
            return candidate

        if not DRIVE_RUN_PREFIX:
            raise ValueError("DRIVE_RUN_PREFIX is required when USE_ADAPTER=True")
        candidates = sorted(
            [
                path for path in DRIVE_RUN_ROOT.glob("*/adapter")
                if path.is_dir() and path.parent.name.startswith(DRIVE_RUN_PREFIX)
            ],
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if not candidates:
            all_runs = sorted(path.parent.name for path in DRIVE_RUN_ROOT.glob("*/adapter") if path.is_dir())
            raise FileNotFoundError(
                f"No adapter folders under {DRIVE_RUN_ROOT} matching prefix {DRIVE_RUN_PREFIX!r}. "
                f"Available adapter runs: {all_runs}"
            )
        return candidates[0]

    adapter_source = find_adapter_dir()
    if LOCAL_ADAPTER_DIR.exists():
        shutil.rmtree(LOCAL_ADAPTER_DIR)
    shutil.copytree(adapter_source, LOCAL_ADAPTER_DIR)
    print("adapter source", adapter_source)
    print("local adapter", LOCAL_ADAPTER_DIR)
    print("files", sorted(path.name for path in LOCAL_ADAPTER_DIR.iterdir()))
else:
    print("USE_ADAPTER=False; loading base model only")


In [ ]:
# Load base model, optionally with LoRA adapter, for inference.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel

print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    token=False,
)
if USE_ADAPTER:
    model = PeftModel.from_pretrained(base_model, str(LOCAL_ADAPTER_DIR))
    print("loaded base model + adapter")
else:
    model = base_model
    print("loaded base model only")
model.eval()
model_context = getattr(model.config, "max_position_embeddings", None)
print("model_context", model_context)
if model_context and MAX_INPUT_TOKENS > model_context:
    raise RuntimeError(f"MAX_INPUT_TOKENS={MAX_INPUT_TOKENS} exceeds model context={model_context}")

generation_config = GenerationConfig.from_model_config(model.config)
generation_config.do_sample = False
generation_config.temperature = None
generation_config.top_p = None
generation_config.top_k = None


In [ ]:
# Prediction helpers.
import json
from build_chiro_sft_jsonl import SYSTEM_PROMPT as SFT_SYSTEM_PROMPT
from build_real_icl_examples import DEFAULT_MANIFEST, build_examples, read_conversation_as_transcript
from claude_few_shot import build_system_prompt as build_icl_system_prompt
from claude_zero_shot import SYSTEM_PROMPT as ZERO_SHOT_SYSTEM_PROMPT
from json_schema import ANTECEDENTE_ENUM, LOCALIZARE_ENUM
from parser import ParseError, SchemaError, parse_note

ICL_EXAMPLE_IDS = [example["conversation_id"] for example in build_examples(DEFAULT_MANIFEST)]
PROMPT_CONFIGS = {
    "sft": (SFT_SYSTEM_PROMPT, "scripts/build_chiro_sft_jsonl.py:SYSTEM_PROMPT"),
    "zero_shot": (ZERO_SHOT_SYSTEM_PROMPT, "src/ICL/claude_zero_shot.py:SYSTEM_PROMPT"),
    "icl": (build_icl_system_prompt(DEFAULT_MANIFEST), "src/ICL/claude_few_shot.py:build_system_prompt(DEFAULT_MANIFEST)"),
}
if PROMPT_PRESET not in PROMPT_CONFIGS:
    raise ValueError(f"Unknown PROMPT_PRESET={PROMPT_PRESET!r}; choose one of {sorted(PROMPT_CONFIGS)}")
SYSTEM_PROMPT, PROMPT_SOURCE = PROMPT_CONFIGS[PROMPT_PRESET]
print("prompt", PROMPT_PRESET, PROMPT_SOURCE)
if PROMPT_PRESET == "icl":
    print("icl examples", ICL_EXAMPLE_IDS)

EMPTY_NOTE = {
    "motivul_prezentarii": None,
    "evaluarea_durerii_vas": None,
    "localizarea_durerii": [],
    "localizarea_durerii_alta": None,
    "antecedente": [],
    "antecedente_altele": None,
    "medicatie_actuala": [],
    "evaluare_functionala_initiala": None,
}

def make_messages(transcript: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "--- TRANSCRIEREA CONSULTAȚIEI ---\n\n" + transcript.strip()},
    ]

def token_count(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def trim_transcript_for_context(transcript: str) -> tuple[str, dict]:
    """Trim transcript before chat templating, so the assistant generation marker is never cut off."""
    empty_prompt = tokenizer.apply_chat_template(
        make_messages(""),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    overhead = token_count(empty_prompt)
    transcript_budget = MAX_INPUT_TOKENS - overhead - 32
    if transcript_budget <= 256:
        raise RuntimeError(
            f"MAX_INPUT_TOKENS={MAX_INPUT_TOKENS} leaves only {transcript_budget} transcript tokens after prompt overhead={overhead}. "
            "For ICL, raise max_input_tokens or reduce examples."
        )

    ids = tokenizer(transcript, add_special_tokens=False)["input_ids"]
    if len(ids) <= transcript_budget:
        return transcript, {
            "transcript_tokens_before": len(ids),
            "transcript_tokens_after": len(ids),
            "trimmed": False,
            "prompt_overhead_tokens": overhead,
            "transcript_budget": transcript_budget,
        }

    head_tokens = min(TRANSCRIPT_HEAD_TOKENS, transcript_budget // 2)
    tail_tokens = transcript_budget - head_tokens
    kept_ids = ids[:head_tokens] + ids[-tail_tokens:]
    trimmed = tokenizer.decode(kept_ids, skip_special_tokens=True)
    return trimmed, {
        "transcript_tokens_before": len(ids),
        "transcript_tokens_after": len(kept_ids),
        "trimmed": True,
        "head_tokens": head_tokens,
        "tail_tokens": tail_tokens,
        "prompt_overhead_tokens": overhead,
        "transcript_budget": transcript_budget,
    }

def generate_raw(transcript: str) -> tuple[str, dict]:
    trimmed_transcript, trim_info = trim_transcript_for_context(transcript)
    prompt = tokenizer.apply_chat_template(
        make_messages(trimmed_transcript),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)
    input_tokens = int(inputs["input_ids"].shape[-1])
    if input_tokens > MAX_INPUT_TOKENS:
        raise RuntimeError(f"Prompt still too long after transcript trimming: {input_tokens} > {MAX_INPUT_TOKENS}")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            generation_config=generation_config,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return raw, {**trim_info, "input_tokens": input_tokens}

def _extract_json_obj(raw: str) -> dict | None:
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        if text.rstrip().endswith("```"):
            text = text.rstrip()[:-3]
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        try:
            value = json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            return None
    return value if isinstance(value, dict) else None

def _nullable_text(value, coercions: list[str], field: str):
    if value is None:
        return None
    if isinstance(value, str):
        stripped = value.strip()
        return stripped or None
    coercions.append(f"{field}: non-string value coerced to null")
    return None

def _append_other(existing, values: list[str]) -> str | None:
    parts = []
    if isinstance(existing, str) and existing.strip():
        parts.append(existing.strip())
    parts.extend(v for v in values if v)
    return "; ".join(parts) if parts else None

def _coerce_enum_list(value, allowed: set[str], other_value, other_field: str, coercions: list[str], field: str):
    kept = []
    invalid = []
    if not isinstance(value, list):
        if value not in (None, []):
            coercions.append(f"{field}: non-list value coerced to []")
        return [], _nullable_text(other_value, coercions, other_field)
    for item in value:
        if not isinstance(item, str):
            coercions.append(f"{field}: non-string item dropped: {item!r}")
            continue
        item = item.strip()
        if item in allowed and item not in kept:
            kept.append(item)
        elif item:
            invalid.append(item)
    if invalid:
        coercions.append(f"{field}: invalid enum values moved to {other_field}: {invalid}")
    return kept, _append_other(_nullable_text(other_value, coercions, other_field), invalid)

def _coerce_vas(value, coercions: list[str]):
    if value is None:
        return None
    if isinstance(value, int) and 0 <= value <= 10:
        return value
    if isinstance(value, list):
        vals = [item for item in value if isinstance(item, int) and 0 <= item <= 10]
        if vals and len(vals) == len(value):
            return vals
    coercions.append(f"evaluarea_durerii_vas: invalid value coerced to null: {value!r}")
    return None

def _coerce_meds(value, coercions: list[str]):
    if value is None:
        return []
    if not isinstance(value, list):
        coercions.append("medicatie_actuala: non-list value coerced to []")
        return []
    meds = []
    for item in value:
        if not isinstance(item, dict):
            coercions.append(f"medicatie_actuala: non-object item dropped: {item!r}")
            continue
        name = item.get("denumire") or item.get("nume")
        if not isinstance(name, str) or not name.strip():
            coercions.append(f"medicatie_actuala: medication without denumire dropped: {item!r}")
            continue
        dose = item.get("doza")
        meds.append({"denumire": name.strip(), "doza": dose.strip() if isinstance(dose, str) and dose.strip() else None})
    return meds

def coerce_prediction_for_eval(parsed: dict | None, error: str | None = None) -> tuple[dict, dict]:
    if parsed is None:
        return dict(EMPTY_NOTE), {
            "prediction_status": "no_parseable_json_saved_empty_note",
            "parse_or_schema_error": error,
            "coercions": ["no parseable JSON object; saved empty schema-valid note so eval can count misses"],
        }

    coercions = []
    note = {
        "motivul_prezentarii": _nullable_text(parsed.get("motivul_prezentarii"), coercions, "motivul_prezentarii"),
        "evaluarea_durerii_vas": _coerce_vas(parsed.get("evaluarea_durerii_vas"), coercions),
        "localizarea_durerii": [],
        "localizarea_durerii_alta": None,
        "antecedente": [],
        "antecedente_altele": None,
        "medicatie_actuala": _coerce_meds(parsed.get("medicatie_actuala"), coercions),
        "evaluare_functionala_initiala": _nullable_text(parsed.get("evaluare_functionala_initiala"), coercions, "evaluare_functionala_initiala"),
    }
    note["localizarea_durerii"], note["localizarea_durerii_alta"] = _coerce_enum_list(
        parsed.get("localizarea_durerii"),
        set(LOCALIZARE_ENUM),
        parsed.get("localizarea_durerii_alta"),
        "localizarea_durerii_alta",
        coercions,
        "localizarea_durerii",
    )
    note["antecedente"], note["antecedente_altele"] = _coerce_enum_list(
        parsed.get("antecedente"),
        set(ANTECEDENTE_ENUM),
        parsed.get("antecedente_altele"),
        "antecedente_altele",
        coercions,
        "antecedente",
    )
    status = "schema_valid" if not coercions and error is None else "coerced_for_eval"
    return note, {
        "prediction_status": status,
        "parse_or_schema_error": error,
        "coercions": coercions,
        "raw_parsed_json": parsed,
    }

def prediction_for_eval(raw: str) -> tuple[dict, dict]:
    try:
        return parse_note(raw), {"prediction_status": "schema_valid", "coercions": []}
    except (ParseError, SchemaError) as exc:
        parsed = _extract_json_obj(raw)
        return coerce_prediction_for_eval(parsed, f"{type(exc).__name__}: {exc}")

def write_note(path: Path, note: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(note, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
# Generate FT condition predictions on TEST_COMPLETE_PAIR_IDS.
failures = []
PRED_DIR.mkdir(parents=True, exist_ok=True)

for idx, conv_id in enumerate(TEST_IDS, start=1):
    out_path = PRED_DIR / f"{conv_id}.json"
    raw_path = PRED_DIR / f"{conv_id}.raw.txt"
    meta_path = PRED_DIR / f"{conv_id}.meta.json"
    if out_path.exists() and not OVERWRITE_PREDICTIONS:
        print(f"[{idx}/{len(TEST_IDS)}] {conv_id} skip")
        continue

    transcript = read_conversation_as_transcript(CONV_DIR / f"{conv_id}.json")
    raw, meta = generate_raw(transcript)
    raw_path.write_text(raw, encoding="utf-8")

    note, pred_meta = prediction_for_eval(raw)
    meta = {**meta, **pred_meta}
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    write_note(out_path, note)

    if meta["prediction_status"] != "schema_valid":
        failures.append({"id": conv_id, "error": meta.get("parse_or_schema_error"), "raw_path": str(raw_path), "meta_path": str(meta_path)})

    trim_msg = "trimmed" if meta.get("trimmed") else "full"
    print(f"[{idx}/{len(TEST_IDS)}] {conv_id} saved ({trim_msg}, {meta['prediction_status']}, input_tokens={meta['input_tokens']})")

(PRED_DIR / "_failures.json").write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
print("done", len(TEST_IDS), "saved", len(failures), "coerced/failed")
print("PRED_DIR", PRED_DIR)


In [ ]:
# Export predictions: zip + Drive backup + optional browser download.
import shutil

if not PRED_DIR.exists():
    raise FileNotFoundError(PRED_DIR)

manifest = {
    "model_preset": MODEL_PRESET,
    "condition_name": CONDITION_NAME,
    "model_id": MODEL_ID,
    "use_adapter": USE_ADAPTER,
    "adapter_source": str(adapter_source) if adapter_source is not None else None,
    "adapter_dir": str(LOCAL_ADAPTER_DIR) if USE_ADAPTER else None,
    "prediction_dir": str(PRED_DIR),
    "test_ids": TEST_IDS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "transcript_head_tokens": TRANSCRIPT_HEAD_TOKENS,
    "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
    "prompt_preset": PROMPT_PRESET,
    "prompt_source": PROMPT_SOURCE,
    "icl_example_ids": ICL_EXAMPLE_IDS if PROMPT_PRESET == "icl" else [],
    "prediction_save_policy": "Always writes <audio>.json. Schema-invalid JSON is coerced for eval and original output/coercions are kept in .raw.txt/.meta.json.",
    "note": "Evaluation is intentionally local-only; this notebook exports predictions/raw outputs/meta only.",
}
(PRED_DIR / "_prediction_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = shutil.make_archive(str(PRED_DIR), "zip", PRED_DIR)
print("prediction zip", zip_path)

# Best-effort Drive backup. If Drive auth fails in extension, use direct download below.
try:
    from google.colab import drive, files  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if Path("/content/drive/MyDrive").exists():
        PREDICTION_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
        drive_zip = PREDICTION_BACKUP_ROOT / f"{CONDITION_NAME}.zip"
        shutil.copy2(zip_path, drive_zip)
        print("Drive prediction zip", drive_zip)
    if DOWNLOAD_PREDICTIONS:
        files.download(zip_path)
except Exception as exc:
    print("Drive/download step failed:", repr(exc))
    print("Zip still exists in Colab at:", zip_path)


## Local Evaluation

După ce aduci zip-ul în repo local, extrage-l astfel încât fișierele să ajungă în directorul condiției, de exemplu:

`data/chiropractor_ro/predictions/cond5_qwen3_4b_synth_claude_ft/`

`data/chiropractor_ro/predictions/cond5_rollama3_8b_synth_claude_ft/`

`data/chiropractor_ro/predictions/cond3_rollama3_8b_real_icl/`

Apoi rulează local, înlocuind `COND_DIR` cu directorul generat:

```bash
python src/SOTA_EVALUATION/eval.py   --pred-dir data/chiropractor_ro/predictions/COND_DIR   --split test-complete   --skip-bertscore
```

Important: notebook-ul nu rulează evaluarea în Colab. Zip-ul conține doar predicții, raw outputs și manifest.
